In [ ]:
# =============================================================
# =============================================================
#
# recupero dati che si riferiscono ai comuni -- 
#
# =============================================================
# =============================================================

In [ ]:
# =============================================================
# =============================================================
#
# librerie di base
import numpy as np
import pandas as pd

In [ ]:
# =============================================================
# =============================================================
#
# librerie fase recupero dati
import requests as api

In [ ]:
# =============================================================
# =============================================================
#
# gestione file e folder e so
import os
import io
from io import StringIO
from os import listdir
from os.path import isfile, join

In [ ]:
def GetFolderDati():
    folderdati = os.getcwd() 
    folderdati =folderdati+ "\\Data" 
    return folderdati

In [ ]:
GetFolderDati()

'c:\\progetto_dataanalyst\\progettofinale\\Data'

In [ ]:
def CancellaFileSeEsiste(fullfilename):
     # verifico se esiste una precedente versione del file
    if os.path.exists(fullfilename):
        print(f"         Il file '{fullfilename}' esiste, per cui lo rimuovo")
        #cancello il file
        os.remove(fullfilename)
    else:
        print(f"      Il file '{fullfilename}' non esiste")

In [ ]:
def SalvaDataset(dati, fullfilename):
   
    CancellaFileSeEsiste(fullfilename)
    
    #salvo il file perchè sarà usato in seguito per tutti i processi successivi
    dati.to_csv(fullfilename)
    print(f"      Il file salvato")

In [ ]:
def SalvaDatasetComuni(dati, nomefile):
    # salvo il dato per fasi successi per evitare di contattare sempre l'api
    file_path_incidente_sorgente = GetFolderDati() +  f'/{nomefile}.csv'
   
    SalvaDataset(dati,file_path_incidente_sorgente)

In [ ]:
def RecuperaESalvaDatiComune(anno):

    #url dati comune porzione base
    urldati_comuni_base="https://situas-servizi.istat.it/publish/reportspooljson?pfun=73&pdata=01/01/"
    #url completo
    urldati_comuni = urldati_comuni_base + str(anno)

    print('   Recupero  dati comune anno '+ str(anno))
    #eseguo la richiesta con Verbo Http GET e con l'aggiunta dell'header nella sezione della richiesta 
    rq_comune_anno = api.get(urldati_comuni)
    
    #lettura risposta
    status_code_risposta= rq_comune_anno.status_code
    print (f"      status_code_risposta : {status_code_risposta}")
    if(status_code_risposta!=200):
        print(f'      errore richiesta api: status_code_risposta : {status_code_risposta}')
    else:
        print('      Converto i dati json ricevuti in DataFrame')
        df_comuni_anno = pd.DataFrame.from_dict(rq_comune_anno.json()['resultset'])

        #aggiungo il riferimento del dataset --importantinssimo
        df_comuni_anno['anno_riferimento'] = anno

        nomefile=f"Dati_Comune_"+ str(anno)
        print(f'      Salvo i dati in CSV su FS. Nome file={nomefile}')
        SalvaDatasetComuni(df_comuni_anno,nomefile)

In [84]:
#considero una finestra temporale di 10 anni.
print("Avvio esportazione dati comune per anno")
anno_esportazione= 2015
while anno_esportazione < 2026:
    print(f"Elaborazione dati comune anno {anno_esportazione}")
    RecuperaESalvaDatiComune(anno_esportazione)
    anno_esportazione= anno_esportazione+1

print("Fine esportazione dati comune per anno")

Avvio esportazione dati comune per anno
Elaborazione dati comune anno 2015
   Recupero  dati comune anno 2015
      status_code_risposta : 200
      Converto i dati json ricevuti in DataFrame
      Salvo i dati in CSV su FS. Nome file=Dati_Comune_2015
         Il file 'c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2015.csv' esiste, per cui lo rimuovo
      Il file salvato
Elaborazione dati comune anno 2016
   Recupero  dati comune anno 2016
      status_code_risposta : 200
      Converto i dati json ricevuti in DataFrame
      Salvo i dati in CSV su FS. Nome file=Dati_Comune_2016
         Il file 'c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2016.csv' esiste, per cui lo rimuovo
      Il file salvato
Elaborazione dati comune anno 2017
   Recupero  dati comune anno 2017
      status_code_risposta : 200
      Converto i dati json ricevuti in DataFrame
      Salvo i dati in CSV su FS. Nome file=Dati_Comune_2017
         Il file 'c:\progetto_dataanalyst\progettofinale\D

In [85]:
#aggregazione dataframe dei comuni
#per errore non lho fatto prima quando recuperavo il dataset per cui lo faccio ora
def UnioneOrizzontaleDataFrame(dataframe_a, dataframe_b):
    return pd.concat([dataframe_a,dataframe_b])

In [86]:
folderdati = GetFolderDati()
#print(f"Cartella dati: {folderdati}")  ok testato

elencoelementi =os.listdir(folderdati)
#print(elencoelementi)  ok testato
elencofiledaconcatenare=[]
for e in elencoelementi:
    #print(e)   ok testato
    fullfilename =os.path.join(folderdati, e)
    if(os.path.isfile(fullfilename)):
       #print("  e' un file")   ok testato
       if (e.startswith("Dati_Comune_20")):
          elencofiledaconcatenare.append(fullfilename)

#print(elencofiledaconcatenare)  ok testato
#print (elencofiledaconcatenare[0]) ok testato

#print(len(elencofiledaconcatenare)) ok testato
risultato = pd.read_csv(elencofiledaconcatenare[0])
for i in range(1, len(elencofiledaconcatenare)):
   #print(i) ok testato
   #print(elencofiledaconcatenare[i])    ok testato
   nuovofile = pd.read_csv(elencofiledaconcatenare[i])
   risultato = pd.concat([risultato,nuovofile])

SalvaDataset(risultato, os.path.join(folderdati, "Dati_Comune_All.csv"))

         Il file 'c:\progetto_dataanalyst\progettofinale\Data\Dati_Comune_All.csv' esiste, per cui lo rimuovo
      Il file salvato
